### g-Ratio Metrics extraction (bilateral analysis)

In [1]:
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = "C:/Users/hapla/Documents/GitHub/albert-plante_project"
csv_path = f"{PROJECT_ROOT}/results/df.csv"
SUBJECT = ["0204", "0261", "0457", "0606", "0733"]
records = []

for sub in SUBJECT:
    RESULTSDIR = f"{PROJECT_ROOT}/results/results_{sub}"
    t1_map_path = f"{RESULTSDIR}/t1_map_mni.nii.gz"
    fa_path = f"{RESULTSDIR}/fa_map_mni.nii.gz"
    fa_tracto_path = f"{RESULTSDIR}/fa_map_tracto_mni.nii.gz"
    fa_mrtrix_path = f"{RESULTSDIR}/fa_map_mrtrix_mni.nii.gz"    
    seg_mp2rage_path = f"{RESULTSDIR}/seg_mp2rage_mni.nii.gz"
    seg_dwi_path = f"{RESULTSDIR}/seg_dwi_mni.nii.gz"

    t1_map = nib.load(t1_map_path).get_fdata()    
    fa_map_scanner = nib.load(fa_path).get_fdata()
    fa_map_tracto = nib.load(fa_tracto_path).get_fdata()
    fa_map_mrtrix = nib.load(fa_mrtrix_path).get_fdata()
    seg_mp2rage = nib.load(seg_mp2rage_path).get_fdata()
    seg_dwi = nib.load(seg_dwi_path).get_fdata()

    mid_x = t1_map.shape[0]//2
    overlap_mask = (seg_mp2rage > 0) & (seg_dwi > 0)

    for y in range(t1_map.shape[1]):
        for side, mask_slice in [("left", slice(0, mid_x)), ("right", slice(mid_x, t1_map.shape[0]))]:
            mask = np.zeros_like(t1_map, dtype=bool)
            mask[mask_slice, y, :] = True
            mask = mask & overlap_mask
            
            fa_scan_vals = fa_map_scanner[mask]
            fa_tracto_vals = fa_map_tracto[mask]
            fa_mrtrix_vals = fa_map_mrtrix[mask]
            t1_vals = t1_map[mask]
            
            for metric, vals in zip(["FA_scanner", "FA_tracto", "FA_mrtrix"], [fa_scan_vals, fa_tracto_vals, fa_mrtrix_vals, t1_vals]):
                if vals.size > 0:
                    records.append({
                        "subject": sub,
                        "slice_index": y,
                        "side": side,
                        "metric": metric,
                        "values": vals.tolist()
                    })
df = pd.DataFrame(records)

df["relative_index"] = None

for (subject, side), group in df.groupby(["subject", "side"]):
    max_slice = group["slice_index"].max()
    rel_idx = max_slice - group["slice_index"]
    df.loc[group.index, "relative_index"] = rel_idx

df = df.drop(columns=["slice_index"])
df = df.rename(columns={"relative_index": "mm"})
df = df[["subject", "side", "mm", "metric", "values"]]
df.loc[df["metric"] == "FA_scanner", "values"] = (df.loc[df["metric"] == "FA_scanner", "values"].apply(lambda vals: [v / 1000 for v in vals]))

df.to_csv(csv_path, index=False)

In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path

csv_path = "C:/Users/hapla/Documents/GitHub/albert-plante_project/results/df_with_MRTRIX.csv"
out_path = "C:/Users/hapla/Documents/GitHub/albert-plante_project/results/df_FA_FVF.csv"

def parse_values(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        try:
            return list(ast.literal_eval(x))
        except Exception:
            try:
                return [float(v) for v in x.strip("[]").split(",") if v.strip()]
            except Exception:
                return []
    return []

def mtvf_from_t1_array(t1_arr):
    t1 = np.array(t1_arr, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        denom = 0.44202 / t1 + 0.94766
        out = 1.0 - 1.0 / denom
    out[~np.isfinite(out)] = np.nan
    return np.clip(out, 0.0, 1.0).tolist()

def mvf_from_mtvf_array(mtvf_arr):
    mvf = 0.5 * np.array(mtvf_arr, dtype=float)
    return np.clip(mvf, 0.0, 1.0).tolist()

def fvf_from_fa_array(fa_arr):
    fa = np.array(fa_arr, dtype=float)
    with np.errstate(invalid="ignore"):
        out = 0.883 * fa**2 - 0.082 * fa + 0.074
    out[~np.isfinite(out)] = np.nan
    return np.clip(out, 0.0, 1.0).tolist()

def gratio_from_mvf_fvf(mvf_arr, fvf_arr):
    n = min(len(mvf_arr), len(fvf_arr))
    if n == 0:
        return []
    mvf = np.array(mvf_arr[:n], dtype=float)
    fvf = np.array(fvf_arr[:n], dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        inner = 1.0 - (mvf / fvf)
        inner[(fvf <= 0) | (~np.isfinite(inner))] = np.nan
        inner[inner < 0] = np.nan
        g = np.sqrt(inner)
    g[~np.isfinite(g)] = np.nan
    return np.clip(g, 0.0, 1.0).tolist()

def key_from_row(row, ignore=("metric", "values")):
    return tuple((c, row[c]) for c in row.index if c not in ignore)

df = pd.read_csv(csv_path)
df["values"] = df["values"].apply(parse_values)

rows_to_add = []

for _, r in df[df["metric"].str.lower() == "t1"].iterrows():
    mtvf_vals = mtvf_from_t1_array(r["values"])  # always seconds
    mvf_vals  = mvf_from_mtvf_array(mtvf_vals)
    base = {c: r[c] for c in df.columns if c not in ("metric", "values")}
    rows_to_add.append({**base, "metric": "MTVF", "values": mtvf_vals})
    rows_to_add.append({**base, "metric": "MVF",  "values": mvf_vals})

for fa_metric, new_metric in (("FA_scanner", "FVF_scanner"), ("FA_tracto", "FVF_tracto"), ("FA_mrtrix", "FVF_mrtrix")):
    for _, r in df[df["metric"] == fa_metric].iterrows():
        fvf_vals = fvf_from_fa_array(r["values"])
        base = {c: r[c] for c in df.columns if c not in ("metric", "values")}
        rows_to_add.append({**base, "metric": new_metric, "values": fvf_vals})

df_aug = pd.concat([df, pd.DataFrame(rows_to_add)], ignore_index=True)

mvf_map    = {key_from_row(r): r["values"] for _, r in df_aug[df_aug["metric"] == "MVF"].iterrows()}
fvf_sc_map = {key_from_row(r): r["values"] for _, r in df_aug[df_aug["metric"] == "FVF_scanner"].iterrows()}
fvf_tr_map = {key_from_row(r): r["values"] for _, r in df_aug[df_aug["metric"] == "FVF_tracto"].iterrows()}

for k, mvf_vals in mvf_map.items():
    key_dict = dict(k)
    if k in fvf_sc_map:
        gvals = gratio_from_mvf_fvf(mvf_vals, fvf_sc_map[k])
        df_aug.loc[len(df_aug)] = {**key_dict, "metric": "g-ratio_scanner", "values": gvals}
    if k in fvf_tr_map:
        gvals = gratio_from_mvf_fvf(mvf_vals, fvf_tr_map[k])
        df_aug.loc[len(df_aug)] = {**key_dict, "metric": "g-ratio_tracto",  "values": gvals}

df_aug = df_aug.sort_values(by=["subject", "side", "mm", "metric"]).reset_index(drop=True)

df_aug.to_csv(out_path, index=False)
print("Saved:", out_path)

### T1, Inv1, Inv2 and UNI extraction

In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd

PROJECT_ROOT = r"C:/Users/hapla/Documents/GitHub/left_right"
RESULTS_ROOT = os.path.join(PROJECT_ROOT, "results")

all_records = []
t1_scaling = {"0204", "0261", "0339", "0457", "0606", "0610", "0733", "0760"}

subj_dirs = [
    d for d in os.listdir(RESULTS_ROOT)
    if d.startswith("sub-") and os.path.isdir(os.path.join(RESULTS_ROOT, d))
]

for subj_dirname in subj_dirs:
    subj = subj_dirname.replace("sub-", "")
    subj_dir = os.path.join(RESULTS_ROOT, subj_dirname)

    inv1_path = os.path.join(subj_dir, f"sub-{subj}_inv1_crop.nii.gz")
    inv2_path = os.path.join(subj_dir, f"sub-{subj}_inv2_crop.nii.gz")
    uni_path  = os.path.join(subj_dir, f"sub-{subj}_uni_crop.nii.gz")
    t1_path   = os.path.join(subj_dir, f"sub-{subj}_t1map_crop.nii.gz")
    seg_path  = os.path.join(subj_dir, "Segmentation.nii")

    try:
        inv1 = nib.load(inv1_path).get_fdata()
        inv2 = nib.load(inv2_path).get_fdata()
        uni  = nib.load(uni_path).get_fdata()
        t1   = nib.load(t1_path).get_fdata()
        seg  = nib.load(seg_path).get_fdata()
    except Exception as e:
        print(f"Missing file in {subj_dirname}: {e}")
        continue

    if subj in t1_scaling:
        t1 = t1 * 1000

    t1_mask = seg > 0
    mid_x = t1.shape[0] // 2

    for y in range(t1.shape[1]):
        left_mask = np.zeros_like(t1, dtype=bool)
        left_mask[mid_x:, y, :] = True
        left_mask &= t1_mask

        right_mask = np.zeros_like(t1, dtype=bool)
        right_mask[:mid_x, y, :] = True
        right_mask &= t1_mask

        for side, mask in [("left", left_mask), ("right", right_mask)]:
            if not np.any(mask):
                continue

            metrics = {
                "Inv1": inv1[mask],
                "Inv2": inv2[mask],
                "UNI":  uni[mask],
                "T1":   t1[mask]
            }

            for metric_name, vals in metrics.items():
                if vals.size > 0:
                    all_records.append({
                        "subject": subj,
                        "slice_index": y,
                        "side": side,
                        "metric": metric_name,
                        "values": vals.tolist()
                    })

df = pd.DataFrame(all_records)

df["mm"] = None
for (subj, side), group in df.groupby(["subject", "side"]):
    max_slice = group["slice_index"].max()
    df.loc[group.index, "mm"] = max_slice - group["slice_index"]

df = df.drop(columns=["slice_index"])
df = df[["subject", "side", "mm", "metric", "values"]]

output_path = os.path.join(RESULTS_ROOT, "all_subjects_df.csv")
df.to_csv(output_path, index=False)

print("Saved:", output_path)

### PAYLOAD creation for HTML file

In [ ]:
import pandas as pd
import json

csv_path = r"C:/Users/hapla/Documents/GitHub/left_right_ds000221/results/all_subjects_df.csv"
df = pd.read_csv(csv_path, converters={"values": eval})

data_list = []
all_subjects = set()
all_metrics = set()
all_mm = []
all_y = []

for _, row in df.iterrows():

    values = row["values"]
    if len(values) == 0:
        continue

    subject = str(row["subject"])
    metric = row["metric"]
    mm_val = float(row["mm"])
    mean_val = float(pd.Series(values).mean())

    entry = {
        "subject": subject,
        "side": row["side"],
        "mm": mm_val,
        "metric": metric,
        "row_mean": mean_val,
        "n": int(len(values)),
    }

    data_list.append(entry)

    all_subjects.add(subject)
    all_metrics.add(metric)
    all_mm.append(mm_val)
    all_y.append(mean_val)

payload = {
    "data": data_list,
    "subjects": sorted(list(all_subjects)),
    "metrics": sorted(list(all_metrics)),
    "mm_min": float(min(all_mm)),
    "mm_max": float(max(all_mm)),
    "y_min": float(min(all_y)),
    "y_max": float(max(all_y)),
}

payload_js = "const PAYLOAD = " + json.dumps(payload, indent=None) + ";"

print(payload_js)